In [ ]:
# ============================================================
# Publication-ready RF feature-importance plots
# Main figure: permutation importance
# Supplement: impurity importance
# Also includes grouped importance summary
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error

# ----------------------------
# PATHS
# ----------------------------
MASTER_CSV = Path(r"C:\goyal_shekhar\post_doc\ecostress\ecostress_master_training_clean.csv")
OUT_DIR = Path(r"C:\goyal_shekhar\post_doc\ecostress\figure_models\rf_importance_pubready")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOPN = 15
N_REPEATS = 30
RANDOM_STATE = 42

# ----------------------------
# MATPLOTLIB STYLE
# ----------------------------
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans"],
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10.5,
    "ytick.labelsize": 10.5,
    "legend.fontsize": 10,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 600,
})

# ----------------------------
# LOAD
# ----------------------------
print(f"Loading: {MASTER_CSV}")
df = pd.read_csv(MASTER_CSV, parse_dates=["time"])
print("Raw shape:", df.shape)

# ----------------------------
# Create T_air lags per station
# ----------------------------
if "station_name" not in df.columns or "time" not in df.columns:
    raise RuntimeError("Need columns 'station_name' and 'time' to create lags safely.")

df = df.sort_values(["station_name", "time"]).copy()

if "T_air" not in df.columns:
    raise RuntimeError("Column 'T_air' is missing. Can't create lags.")

for k in range(1, 6):
    df[f"T_air_lag{k}"] = df.groupby("station_name")["T_air"].shift(k)

# ----------------------------
# FEATURE LIST
# ----------------------------
feature_cols = [
    "center_mean",
    "buf100m_mean",

    "Q",
    "T_air",
    "T_air_lag1",
    "T_air_lag2",
    "T_air_lag3",
    "T_air_lag4",
    "T_air_lag5",

    "DEM_m",
    "slope_deg_or_pct",
    "lulc_10",
    "lulc_30",
    "lulc_40",
    "lulc_50",
    "lulc_60",

    "doy_sin",
    "doy_cos",
]

target_col = "T_w_obs"

# ----------------------------
# Fill short buffers with center if needed
# ----------------------------
for r in [50, 100, 150]:
    col = f"buf{r}m_mean"
    if col in df.columns and "center_mean" in df.columns:
        df[col] = df[col].fillna(df["center_mean"])

missing = [c for c in feature_cols + [target_col] if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing columns in CSV: {missing}")

df = df.dropna(subset=feature_cols + [target_col]).copy()
print("After lagging + dropna:", df.shape)
print("Stations retained:", df["station_name"].nunique())

# ----------------------------
# TRAIN / TEST SPLIT
# ----------------------------
X = df[feature_cols].copy()
y = df[target_col].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

print("Training RF...")
rf.fit(X_train, y_train)

base_rmse = np.sqrt(mean_squared_error(y_test, rf.predict(X_test)))
print(f"Test baseline RMSE: {base_rmse:.3f} °C")

# ----------------------------
# LABELS
# ----------------------------
def nice_label(s: str) -> str:
    rep = {
        "center_mean": "ECOSTRESS LST (center 3×3)",
        "buf100m_mean": "ECOSTRESS LST (100 m buffer)",
        "DEM_m": "Elevation",
        "slope_deg_or_pct": "Slope",
        "Q": "Discharge",
        "T_air": "Air temperature (t)",
        "T_air_lag1": "Air temperature (t−1)",
        "T_air_lag2": "Air temperature (t−2)",
        "T_air_lag3": "Air temperature (t−3)",
        "T_air_lag4": "Air temperature (t−4)",
        "T_air_lag5": "Air temperature (t−5)",
        "doy_sin": "Seasonality (sin DOY)",
        "doy_cos": "Seasonality (cos DOY)",
        "lulc_10": "Tree cover",
        "lulc_30": "Grassland",
        "lulc_40": "Cropland",
        "lulc_50": "Built-up",
        "lulc_60": "Bare ground",
    }
    return rep.get(s, s.replace("_", " "))

def feature_group(s: str) -> str:
    if s in ["center_mean", "buf100m_mean"]:
        return "ECOSTRESS"
    elif s == "Q":
        return "Hydrology"
    elif s.startswith("T_air"):
        return "Air temperature"
    elif s in ["doy_sin", "doy_cos"]:
        return "Seasonality"
    elif s in ["DEM_m", "slope_deg_or_pct", "lulc_10", "lulc_30", "lulc_40", "lulc_50", "lulc_60"]:
        return "Static landscape"
    else:
        return "Other"

group_color = {
    "ECOSTRESS": "#1f4e79",
    "Air temperature": "#7f7f7f",
    "Hydrology": "#4f81bd",
    "Seasonality": "#bdbdbd",
    "Static landscape": "#9e9e9e",
    "Other": "#cccccc",
}

# ----------------------------
# IMPORTANCE CALCULATIONS
# ----------------------------
# Impurity importance
impurity = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
#impurity.to_csv(OUT_DIR / "rf_importance_impurity_full.csv")

# Permutation importance
perm = permutation_importance(
    rf,
    X_test,
    y_test,
    scoring="neg_mean_squared_error",
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# IMPORTANT: do NOT negate these
mda_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
})
mda_df["group"] = mda_df["feature"].map(feature_group)
mda_df = mda_df.sort_values("importance_mean", ascending=False)
#mda_df.to_csv(OUT_DIR / "rf_importance_permutation_full.csv", index=False)

# ----------------------------
# MAIN PANEL: permutation importance
# ----------------------------
top = mda_df.head(TOPN).copy().iloc[::-1]
labels = [nice_label(x) for x in top["feature"]]
colors = [group_color[g] for g in top["group"]]

fig, ax = plt.subplots(figsize=(8.8, 6.2))
bars = ax.barh(
    labels,
    top["importance_mean"].values,
    xerr=top["importance_std"].values,
    color=colors,
    edgecolor="black",
    linewidth=0.5,
    capsize=3
)

# subtle hatching for ECOSTRESS
for bar, grp in zip(bars, top["group"]):
    if grp == "ECOSTRESS":
        bar.set_hatch("///")

ax.set_xlabel("Decrease in model skill after permutation")
ax.set_ylabel("")
ax.set_title("Predictor importance in the Random Forest model")
ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

# compact legend
from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor=group_color[g], edgecolor="black",
          hatch="///" if g == "ECOSTRESS" else None, label=g)
    for g in ["ECOSTRESS", "Air temperature", "Hydrology", "Seasonality", "Static landscape"]
]
ax.legend(handles=legend_handles, frameon=False, loc="lower right")

plt.tight_layout()
#plt.savefig(OUT_DIR / "rf_importance_permutation_top.png", bbox_inches="tight")
plt.show()

# ----------------------------
# GROUPED IMPORTANCE PANEL
# ----------------------------
grouped = (
    mda_df.groupby("group", as_index=True)["importance_mean"]
    .sum()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(6.8, 4.3))
ax.barh(
    grouped.index,
    grouped.values,
    color=[group_color[g] for g in grouped.index],
    edgecolor="black",
    linewidth=0.5
)

for i, v in enumerate(grouped.values):
    ax.text(v + 0.01 * grouped.max(), i, f"{v:.3f}", va="center", fontsize=10)

ax.set_xlabel("Summed permutation importance")
ax.set_ylabel("")
ax.set_title("Grouped predictor importance")
ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(OUT_DIR / "rf_importance_grouped.png", bbox_inches="tight")
plt.show()

# ----------------------------
# SUPPLEMENT: impurity importance
# ----------------------------
imp_df = impurity.reset_index()
imp_df.columns = ["feature", "importance"]
imp_df["group"] = imp_df["feature"].map(feature_group)

top_imp = imp_df.head(TOPN).copy().iloc[::-1]
labels_imp = [nice_label(x) for x in top_imp["feature"]]
colors_imp = [group_color[g] for g in top_imp["group"]]

fig, ax = plt.subplots(figsize=(8.8, 6.2))
bars = ax.barh(
    labels_imp,
    top_imp["importance"].values,
    color=colors_imp,
    edgecolor="black",
    linewidth=0.5
)

for bar, grp in zip(bars, top_imp["group"]):
    if grp == "ECOSTRESS":
        bar.set_hatch("///")

ax.set_xlabel("Impurity-based importance")
ax.set_ylabel("")
ax.set_title("Predictor importance in the Random Forest model")
ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
#plt.savefig(OUT_DIR / "rf_importance_impurity_top.png", bbox_inches="tight")
plt.show()

print("\nSaved outputs to:")
print(OUT_DIR)